Import necessary library for EDA and Data Cleaning

In [1]:
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
import seaborn as sns

Scan through CSV and collect the first 10 rows.

In [2]:
df = (
    pl
    .scan_csv('uci-secom.csv', has_header=True)
    .with_columns(pl.col('Time').str.strptime(dtype=pl.Datetime, format='%Y-%m-%d %H:%M:%S'))
)
print(df.head(10).collect())

shape: (10, 592)
┌─────────────────────┬─────────┬─────────┬───────────┬───┬────────┬────────┬──────────┬───────────┐
│ Time                ┆ 0       ┆ 1       ┆ 2         ┆ … ┆ 587    ┆ 588    ┆ 589      ┆ Pass/Fail │
│ ---                 ┆ ---     ┆ ---     ┆ ---       ┆   ┆ ---    ┆ ---    ┆ ---      ┆ ---       │
│ datetime[μs]        ┆ f64     ┆ f64     ┆ f64       ┆   ┆ f64    ┆ f64    ┆ f64      ┆ i64       │
╞═════════════════════╪═════════╪═════════╪═══════════╪═══╪════════╪════════╪══════════╪═══════════╡
│ 2008-07-19 11:55:00 ┆ 3030.93 ┆ 2564.0  ┆ 2187.7333 ┆ … ┆ null   ┆ null   ┆ null     ┆ -1        │
│ 2008-07-19 12:32:00 ┆ 3095.78 ┆ 2465.14 ┆ 2230.4222 ┆ … ┆ 0.0201 ┆ 0.006  ┆ 208.2045 ┆ -1        │
│ 2008-07-19 13:17:00 ┆ 2932.61 ┆ 2559.94 ┆ 2186.4111 ┆ … ┆ 0.0484 ┆ 0.0148 ┆ 82.8602  ┆ 1         │
│ 2008-07-19 14:43:00 ┆ 2988.72 ┆ 2479.9  ┆ 2199.0333 ┆ … ┆ 0.0149 ┆ 0.0044 ┆ 73.8432  ┆ -1        │
│ 2008-07-19 15:22:00 ┆ 3032.24 ┆ 2502.87 ┆ 2233.3667 ┆ … ┆ 0.0149 ┆ 0.004

Check for Dtypes inside the CSV

In [3]:
from collections import Counter

schema = df.collect_schema()
for dtype, count in Counter(schema.values()).items():
    print(f'{dtype}: {count}')

Datetime(time_unit='us', time_zone=None): 1
Float64: 574
String: 16
Int64: 1


Check what the Column with string Datatypes Contain

In [4]:
is_string = (
    df
    .select(cs.string())
)
with pl.Config(tbl_cols=16):
    print(is_string.head().collect())

shape: (5, 16)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬──────┐
│ 85  ┆ 109 ┆ 110 ┆ 111 ┆ 220 ┆ 244 ┆ 245 ┆ 246 ┆ 358 ┆ 382 ┆ 383 ┆ 384 ┆ 492 ┆ 516 ┆ 517 ┆ 518  │
│ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ ---  │
│ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str ┆ str  │
╞═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪═════╪══════╡
│ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ null │
│ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆      │
│ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ null │
│ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆ l   ┆      │
│ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nul ┆ nu

Since The string Dtype Columns are all filled with null, we are going to drop all of them.

In [5]:
df = (
    df
    .drop(cs.string())
)

print(f'Non-String Column Left: {df.collect_schema().len()}')

Non-String Column Left: 576


Counting through the average of null per row. Since it was only an average of 15 null value per row, We are going to fill it with mean of the column later on.

In [6]:
avg_null_per_row = (
    df
    .with_columns(null_count = pl.sum_horizontal(pl.all().is_null()))
    .select(pl.col('null_count').mean())
    .collect()
    .item()
)

print(avg_null_per_row)

15.552648372686662


I also noticed that there are some cuplicated columns inside the dataset, so I decided to drop all of them and only keep the first occurence

In [7]:
cols = df.collect_schema().names()
sig_row = (
    df
    .select([pl.col(c).implode().alias(c) for c in cols])
    .collect()
    .row(0)
)

seen = set()
keep = []

for name, values in zip(cols, sig_row):
    key = tuple(values)
    if key not in seen:
        seen.add(key)
        keep.append(name)

df = (df.select(keep))
print(f'Duplicated columns dropped: {576 - df.collect().shape[1]} column')

Duplicated columns dropped: 104 column


Changing the -1/1 Value to 0/1 for Classification models to work with.

In [8]:
df = (
    df
    .with_columns(
        pl
        .when(pl.col('Pass/Fail') == -1)
        .then(pl.lit(0))
        .otherwise(pl.lit(1))
        .alias('Pass/Fail')
    )
)

- Importing necessary libraries for Classification model
- Importing necessary libraries for Data preprocessing
- Assigning Model and Preprocessor to Variables

In [9]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline

X = df.select(cs.float()).collect().to_numpy()
y = df.select(cs.integer()).collect().to_numpy().ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

xgb = XGBClassifier(scale_pos_weight=14)

rf = RandomForestClassifier(max_features='sqrt', class_weight='balanced')

preprocessor_xgb = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('variance', VarianceThreshold()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('scaler', StandardScaler())
])

preprocessor_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('variance', VarianceThreshold()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('scaler', StandardScaler())
])


xgb_pipe = Pipeline([
    ('preprocessing', preprocessor_xgb),
    ('model', xgb)
])

rf_pipe = Pipeline([
    ('preprocessing', preprocessor_rf),
    ('model', rf)
])

Setting up a parameter grid for GridSearchCV with f1 scoring method

In [10]:
xgb_param_prid = {
    'preprocessing__selector__k' : [10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100],
    'model__booster' : ['gbtree', 'dart'],
    'model__eta' : [0.1, 0.2, 0.3, 0.4, 0.5],
    'model__max_depth' : [6, 10, 15, 20]
}

rf_param_grid = {
    'preprocessing__selector__k' : [10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100],
    'model__n_estimators' : [100, 200, 300, 500],
    'model__max_depth' : [6, 10, 15, 20],
}

xgb_grid = GridSearchCV(xgb_pipe, xgb_param_prid, cv=5, scoring='f1')
rf_grid = GridSearchCV(rf_pipe, rf_param_grid, cv=5, scoring='f1')

Fitting the XGB CVGridSerach and printing it's best parameter and F1 score.

In [11]:
xgb_grid.fit(X_train, y_train)
print(xgb_grid.best_params_)
print(xgb_grid.best_score_)

{'model__booster': 'gbtree', 'model__eta': 0.4, 'model__max_depth': 10, 'preprocessing__selector__k': 70}
0.1796103896103896


Fitting the RF CVGridSerach and printing it's best parameter and F1 score.

In [13]:
rf_grid.fit(X_train, y_train)
print(rf_grid.best_params_)
print(rf_grid.best_score_)

{'model__max_depth': 6, 'model__n_estimators': 100, 'preprocessing__selector__k': 35}
0.17716317665343154


From the GridSearch we can conclude that XGB is the best model for this dataset.
With the parameter mentioned above, we will create our final pipeline and search for the best threshold.

In [49]:
from sklearn.metrics import classification_report, precision_recall_curve, f1_score

my_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('variance', VarianceThreshold()),
    ('selector', SelectKBest(score_func=f_classif, k=70)),
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(booster='gbtree', eta=0.4, max_depth=10, scale_pos_weight=14))
])

my_pipe.fit(X_train, y_train)
y_proba = my_pipe.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8) #Adding 1e-8 to avoid zero division error.
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.3f}")
print(f"F1 at that threshold: {f1_scores[best_idx]:.3f}")

Best threshold: 0.236
F1 at that threshold: 0.364


We can see here that with the parameter and threshold we use, we can get up to 93% of accuracy and 0.36 on the minority class f1-score.

In [50]:
y_pred = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.95      0.98      0.96       293
           1       0.50      0.29      0.36        21

    accuracy                           0.93       314
   macro avg       0.73      0.63      0.66       314
weighted avg       0.92      0.93      0.92       314



Creating a class to apply threshold to the model.

In [51]:
class ThresholdedClassifier:
    def __init__(self, pipeline, threshold):
        self.pipeline = pipeline
        self.threshold = threshold
    
    def predict(self, X):
        proba = self.pipeline.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)
    
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

final_pipe = ThresholdedClassifier(my_pipe, 0.236)

Saving model.

In [52]:
import joblib
joblib.dump(final_pipe, 'finalpipe.joblib')

['finalpipe.joblib']